In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visualization style for all plots
sns.set_theme(style="whitegrid", palette="muted")

# Define the directory where your 17 CSV files are stored
DATA_DIR = '../data/PRO-ACT_Data/2026_02_27_PROACT_ALL_FORMS/' 

# ---------------------------------------------------------
# 1. DATA LOADING
# ---------------------------------------------------------
def load_data(filename):
    """Safely load a CSV file into a pandas DataFrame."""
    file_path = os.path.join(DATA_DIR, filename)
    try:
        print(f"Loading {filename}...")
        return pd.read_csv(file_path, low_memory=False)
    except FileNotFoundError:
        print(f"  -> Warning: {filename} not found. Skipping.")
        return None

print("=== PART 1: LOADING DATASETS ===")
datasets = {
    'demographics': load_data('F_PROACT_DEMOGRAPHICS.csv'),
    'alsfrs': load_data('F_PROACT_ALSFRS.csv'),
    'death': load_data('F_PROACT_DEATHDATA.csv'),
    'treatment': load_data('F_PROACT_TREATMENT.csv'),
    'riluzole': load_data('F_PROACT_RILUZOLE.csv'),
    'fvc': load_data('F_PROACT_FVC.csv'),
    'svc': load_data('F_PROACT_SVC.csv'),
    'vitals': load_data('F_PROACT_VITALSIGNS.csv'),
    'handgrip': load_data('F_PROACT_HANDGRIPSTRENGTH.csv'),
    'muscle': load_data('F_PROACT_MUSCLESTRENGTH.csv'),
    'neurofilament': load_data('F_PROACT_Neurofilament.csv'),
    'adverse_events': load_data('F_PROACT_ADVERSEEVENTS.csv'),
    'history': load_data('F_PROACT_ALSHISTORY.csv'),
    'conmeds': load_data('F_PROACT_CONMEDS.csv'),
    'elescorial': load_data('F_PROACT_ELESCORIAL.csv'),
    'family': load_data('F_PROACT_FAMILYHISTORY.csv'),
    'labs': load_data('F_PROACT_LABS.csv')
}

# ---------------------------------------------------------
# 2. DATA CLEANING & STANDARDIZATION
# ---------------------------------------------------------
print("\n=== PART 2: CLEANING AND STANDARDIZING DATA ===")

if datasets['demographics'] is not None:
    datasets['demographics']['Age'] = pd.to_numeric(datasets['demographics']['Age'], errors='coerce')
    datasets['demographics']['Sex'] = datasets['demographics']['Sex'].astype(str).str.strip().str.title()

if datasets['alsfrs'] is not None:
    df = datasets['alsfrs']
    for col in ['ALSFRS_Delta', 'ALSFRS_Total']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    datasets['alsfrs'] = df.dropna(subset=['ALSFRS_Delta'], how='all')

if datasets['death'] is not None:
    datasets['death']['Death_Days'] = pd.to_numeric(datasets['death']['Death_Days'], errors='coerce')
    datasets['death']['Status'] = datasets['death']['Subject_Died'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

if datasets['treatment'] is not None:
    datasets['treatment']['Study_Arm'] = datasets['treatment']['Study_Arm'].astype(str).str.strip().str.title()

if datasets['riluzole'] is not None:
    datasets['riluzole']['Subject_used_Riluzole'] = datasets['riluzole']['Subject_used_Riluzole'].astype(str).str.strip().str.title()

if datasets['history'] is not None:
    datasets['history']['Site_of_Onset'] = datasets['history']['Site_of_Onset'].astype(str).str.strip().str.title()

if datasets['fvc'] is not None:
    datasets['fvc']['pct_of_Normal_Trial_1'] = pd.to_numeric(datasets['fvc']['pct_of_Normal_Trial_1'], errors='coerce')

if datasets['neurofilament'] is not None:
    datasets['neurofilament']['Result'] = pd.to_numeric(datasets['neurofilament']['Result'], errors='coerce')

if datasets['vitals'] is not None:
    datasets['vitals']['Weight'] = pd.to_numeric(datasets['vitals']['Weight'], errors='coerce')

if datasets['handgrip'] is not None:
    datasets['handgrip']['Test_Result'] = pd.to_numeric(datasets['handgrip']['Test_Result'], errors='coerce')
    datasets['handgrip']['MS_Delta'] = pd.to_numeric(datasets['handgrip']['MS_Delta'], errors='coerce')

if datasets['labs'] is not None:
    datasets['labs']['Test_Result'] = pd.to_numeric(datasets['labs']['Test_Result'], errors='coerce')
    datasets['labs']['Laboratory_Delta'] = pd.to_numeric(datasets['labs']['Laboratory_Delta'], errors='coerce')
    datasets['labs']['Test_Name'] = datasets['labs']['Test_Name'].astype(str).str.strip().str.upper()

# ---------------------------------------------------------
# 3. COMPREHENSIVE DATA EXPLORATION
# ---------------------------------------------------------
print("\n=== PART 3: DATA EXPLORATION (SUMMARY) ===")

def explore_dataframe(df, name):
    if df is not None:
        print(f"--- {name.upper()} ---")
        print(f"Total Rows: {df.shape[0]:,} | Total Columns: {df.shape[1]}")
        if 'subject_id' in df.columns:
            print(f"Unique Subjects: {df['subject_id'].nunique():,}")
        
        # Calculate sparsity (percentage of missing values)
        # FIXED: Using df.size instead of the deprecated np.product
        total_cells = df.size
        missing_cells = df.isnull().sum().sum()
        sparsity = (missing_cells / total_cells) * 100 if total_cells > 0 else 0
        print(f"Dataset Sparsity: {sparsity:.1f}% missing values overall")
        print("-" * 35)

for name, df in datasets.items():
    explore_dataframe(df, name)

# ---------------------------------------------------------
# 4. EXTENSIVE VISUALIZATIONS
# ---------------------------------------------------------
print("\n=== PART 4: GENERATING VISUALIZATIONS ===")
rlt_dir = '../PROACT_results/'
if not os.path.exists(rlt_dir):
    os.makedirs(rlt_dir)

# 1. Demographics: Age and Sex Distribution
if datasets['demographics'] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.histplot(datasets['demographics']['Age'].dropna(), bins=25, kde=True, color='#3498db', ax=axes[0])
    axes[0].set_title('Distribution of Age at Trial Entry', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Age (Years)')
    
    sns.countplot(data=datasets['demographics'], x='Sex', palette='pastel', ax=axes[1], order=['Male', 'Female'])
    axes[1].set_title('Distribution of Biological Sex', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(rlt_dir + '01_Demographics.png')
    plt.close()

# 2. ALS History: Site of Onset
if datasets['history'] is not None:
    plt.figure(figsize=(8, 5))
    valid_onset = datasets['history'][datasets['history']['Site_of_Onset'].str.contains('Onset', na=False)]
    if not valid_onset.empty:
        sns.countplot(data=valid_onset, y='Site_of_Onset', palette='viridis', order=valid_onset['Site_of_Onset'].value_counts().index)
        plt.title('Patient Site of Disease Onset', fontsize=12, fontweight='bold')
        plt.xlabel('Number of Patients')
        plt.ylabel('')
        plt.tight_layout()
        plt.savefig(rlt_dir + '02_Site_of_Onset.png')
    plt.close()

# 3. Treatment & Riluzole Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if datasets['treatment'] is not None and not datasets['treatment'].empty:
    t_counts = datasets['treatment']['Study_Arm'].value_counts()
    axes[0].pie(t_counts, labels=t_counts.index, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#f1c40f'])
    axes[0].set_title('Study Arm Assignment', fontweight='bold')
if datasets['riluzole'] is not None and not datasets['riluzole'].empty:
    r_counts = datasets['riluzole']['Subject_used_Riluzole'].value_counts()
    axes[1].pie(r_counts, labels=r_counts.index, autopct='%1.1f%%', colors=['#9b59b6', '#bdc3c7'])
    axes[1].set_title('Baseline Riluzole Usage', fontweight='bold')
plt.savefig(rlt_dir + '03_Treatment_Riluzole.png')
plt.close()

# 4. Disease Progression: ALSFRS Score over First 2 Years
if datasets['alsfrs'] is not None and not datasets['alsfrs'].empty:
    df_als = datasets['alsfrs']
    df_als_filt = df_als[(df_als['ALSFRS_Delta'] >= 0) & (df_als['ALSFRS_Delta'] <= 730)].copy()
    df_als_filt['Month'] = (df_als_filt['ALSFRS_Delta'] / 30).astype(int)
    monthly_trend = df_als_filt.groupby('Month')['ALSFRS_Total'].mean().reset_index()

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=monthly_trend, x='Month', y='ALSFRS_Total', marker='o', color='#e67e22', linewidth=2.5)
    plt.title('Average ALSFRS Total Score Decline (First 2 Years)', fontsize=14, fontweight='bold')
    plt.xlabel('Months Since Trial Start')
    plt.ylabel('Average ALSFRS Total Score')
    plt.fill_between(monthly_trend['Month'], monthly_trend['ALSFRS_Total'] - 2, monthly_trend['ALSFRS_Total'] + 2, color='#e67e22', alpha=0.2)
    plt.savefig(rlt_dir + '04_ALSFRS_Progression.png')
    plt.close()

# 5. Respiratory Function: Baseline FVC Distribution
if datasets['fvc'] is not None and not datasets['fvc'].empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(datasets['fvc']['pct_of_Normal_Trial_1'].dropna(), bins=30, color='#1abc9c', kde=True)
    plt.title('Baseline Forced Vital Capacity (FVC) % of Normal', fontsize=12, fontweight='bold')
    plt.xlabel('FVC (% of Normal)')
    plt.xlim(0, 150)
    plt.axvline(x=80, color='red', linestyle='--', label='Clinical Threshold (80%)')
    plt.legend()
    plt.savefig(rlt_dir + '05_FVC_Baseline.png')
    plt.close()

# 6. Biomarkers: Neurofilament 
if datasets['neurofilament'] is not None and not datasets['neurofilament'].empty:
    plt.figure(figsize=(8, 5))
    nf_95 = datasets['neurofilament']['Result'].quantile(0.95)
    clean_nf = datasets['neurofilament'][datasets['neurofilament']['Result'] <= nf_95]['Result'].dropna()
    sns.histplot(clean_nf, bins=40, color='#9b59b6', kde=True)
    plt.title('Neurofilament Light Chain Levels (pg/mL)', fontsize=12, fontweight='bold')
    plt.xlabel('Result (pg/mL)')
    plt.savefig(rlt_dir + '06_Neurofilament.png')
    plt.close()

# 7. Strength: Handgrip Over Time
if datasets['handgrip'] is not None and not datasets['handgrip'].empty:
    df_hg = datasets['handgrip']
    df_hg_filt = df_hg[(df_hg['MS_Delta'] >= 0) & (df_hg['MS_Delta'] <= 540)].copy()
    df_hg_filt['Month'] = (df_hg_filt['MS_Delta'] / 30).astype(int)
    monthly_hg = df_hg_filt.groupby('Month')['Test_Result'].mean().reset_index()

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=monthly_hg, x='Month', y='Test_Result', marker='s', color='#27ae60', linewidth=2)
    plt.title('Average Handgrip Strength Decline (First 1.5 Years)', fontsize=12, fontweight='bold')
    plt.xlabel('Months Since Trial Start')
    plt.ylabel('Average Grip Strength (Pounds)')
    plt.savefig(rlt_dir + '07_Handgrip_Decline.png')
    plt.close()

# 8. Safety: Top 10 Most Common Adverse Events
if datasets['adverse_events'] is not None and not datasets['adverse_events'].empty:
    plt.figure(figsize=(10, 6))
    top_aes = datasets['adverse_events']['Preferred_Term'].value_counts().head(10)
    sns.barplot(x=top_aes.values, y=top_aes.index, palette='Reds_r')
    plt.title('Top 10 Most Frequent Adverse Events Reported', fontsize=12, fontweight='bold')
    plt.xlabel('Total Number of Occurrences')
    plt.tight_layout()
    plt.savefig(rlt_dir + '08_Adverse_Events.png')
    plt.close()

print("--> 8 visualization PNG files successfully saved to your directory.")

# ---------------------------------------------------------
# 5. FEATURE ENGINEERING & MASTER DATASET MERGE
# ---------------------------------------------------------
print("\n=== PART 5: BUILDING MASTER DATASET FOR ML ===")

# Initialize Master DF with unique subjects
master_df = pd.DataFrame()
if datasets['demographics'] is not None:
    master_df = pd.DataFrame({'subject_id': datasets['demographics']['subject_id'].unique()})

def merge_static_feature(master, df, columns):
    """Merges single-value (static) datasets into the master table."""
    if df is not None and not df.empty:
        # Drop duplicates to ensure a 1:1 merge per subject
        unique_df = df[['subject_id'] + columns].drop_duplicates(subset=['subject_id'])
        return pd.merge(master, unique_df, on='subject_id', how='left')
    return master

# Merge core static features
master_df = merge_static_feature(master_df, datasets['demographics'], ['Age', 'Sex'])
master_df = merge_static_feature(master_df, datasets['death'], ['Status', 'Death_Days'])
master_df = merge_static_feature(master_df, datasets['treatment'], ['Study_Arm'])
master_df = merge_static_feature(master_df, datasets['riluzole'], ['Subject_used_Riluzole'])
master_df = merge_static_feature(master_df, datasets['elescorial'], ['el_escorial'])
master_df = merge_static_feature(master_df, datasets['history'], ['Site_of_Onset'])

def get_baseline_longitudinal(df, delta_col, value_col, new_col_name):
    """Filters longitudinal data for the first ~30 days to establish a baseline average."""
    if df is not None and not df.empty and delta_col in df.columns and value_col in df.columns:
        baseline_data = df[(df[delta_col] >= 0) & (df[delta_col] <= 30)].copy()
        baseline_grouped = baseline_data.groupby('subject_id')[value_col].mean().reset_index()
        baseline_grouped.rename(columns={value_col: new_col_name}, inplace=True)
        return baseline_grouped
    return pd.DataFrame(columns=['subject_id', new_col_name])

# Merge Baseline Features from Time-Series Data
master_df = pd.merge(master_df, get_baseline_longitudinal(datasets['alsfrs'], 'ALSFRS_Delta', 'ALSFRS_Total', 'Baseline_ALSFRS'), on='subject_id', how='left')
master_df = pd.merge(master_df, get_baseline_longitudinal(datasets['fvc'], 'Forced_Vital_Capacity_Delta', 'pct_of_Normal_Trial_1', 'Baseline_FVC_pct'), on='subject_id', how='left')
master_df = pd.merge(master_df, get_baseline_longitudinal(datasets['vitals'], 'Vital_Signs_Delta', 'Weight', 'Baseline_Weight'), on='subject_id', how='left')

# Handle Large Lab Data
if datasets['labs'] is not None and not datasets['labs'].empty:
    df_labs = datasets['labs']
    baseline_labs = df_labs[(df_labs['Laboratory_Delta'] >= 0) & (df_labs['Laboratory_Delta'] <= 30)]
    
    key_labs = ['CREATININE', 'HEMOGLOBIN', 'WHITE BLOOD CELL (WBC)', 'ALKALINE PHOSPHATASE', 'SODIUM']
    filtered_labs = baseline_labs[baseline_labs['Test_Name'].isin(key_labs)]
    
    pivoted_labs = filtered_labs.pivot_table(index='subject_id', columns='Test_Name', values='Test_Result', aggfunc='mean').reset_index()
    pivoted_labs.columns = ['subject_id'] + [f"Lab_{str(col).replace(' ', '_')}" for col in pivoted_labs.columns[1:]]
    master_df = pd.merge(master_df, pivoted_labs, on='subject_id', how='left')

# ---------------------------------------------------------
# 6. FINAL REVIEW & EXPORT
# ---------------------------------------------------------
print("\n=== PART 6: EXPORT ===")
if not master_df.empty:
    print(f"Master Dataset Shape: {master_df.shape[0]:,} Rows (Patients) x {master_df.shape[1]} Columns (Features)")
    
    output_filename = 'PROACT_Master_Dataset.csv'
    master_df.to_csv(rlt_dir + output_filename, index=False)
    print(f"\nSUCCESS: Master dataset saved to '{output_filename}'")
else:
    print("Master dataset is empty. Ensure demographics data is available.")

=== PART 1: LOADING DATASETS ===
Loading F_PROACT_DEMOGRAPHICS.csv...
Loading F_PROACT_ALSFRS.csv...
Loading F_PROACT_DEATHDATA.csv...
Loading F_PROACT_TREATMENT.csv...
Loading F_PROACT_RILUZOLE.csv...
Loading F_PROACT_FVC.csv...
Loading F_PROACT_SVC.csv...
Loading F_PROACT_VITALSIGNS.csv...
Loading F_PROACT_HANDGRIPSTRENGTH.csv...
Loading F_PROACT_MUSCLESTRENGTH.csv...
Loading F_PROACT_Neurofilament.csv...
Loading F_PROACT_ADVERSEEVENTS.csv...
Loading F_PROACT_ALSHISTORY.csv...
Loading F_PROACT_CONMEDS.csv...
Loading F_PROACT_ELESCORIAL.csv...
Loading F_PROACT_FAMILYHISTORY.csv...
Loading F_PROACT_LABS.csv...

=== PART 2: CLEANING AND STANDARDIZING DATA ===

=== PART 3: DATA EXPLORATION (SUMMARY) ===
--- DEMOGRAPHICS ---
Total Rows: 13,115 | Total Columns: 14
Unique Subjects: 13,115
Dataset Sparsity: 63.5% missing values overall
-----------------------------------
--- ALSFRS ---
Total Rows: 81,041 | Total Columns: 20
Unique Subjects: 9,149
Dataset Sparsity: 28.0% missing values overal

/tmp/ipykernel_1691437/2061429965.py:133: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=datasets['demographics'], x='Sex', palette='pastel', ax=axes[1], order=['Male', 'Female'])
/tmp/ipykernel_1691437/2061429965.py:145: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=valid_onset, y='Site_of_Onset', palette='viridis', order=valid_onset['Site_of_Onset'].value_counts().index)
/tmp/ipykernel_1691437/2061429965.py:224: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_aes.values, y=top_aes.index, palette='Reds_r')


--> 8 visualization PNG files successfully saved to your directory.

=== PART 5: BUILDING MASTER DATASET FOR ML ===

=== PART 6: EXPORT ===
Master Dataset Shape: 13,115 Rows (Patients) x 17 Columns (Features)

SUCCESS: Master dataset saved to 'PROACT_Master_Dataset.csv'


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visualization style for all plots
sns.set_theme(style="whitegrid", palette="muted")

# Define the directory where your 17 CSV files are stored
DATA_DIR = '../data/PRO-ACT_Data/2026_02_27_PROACT_ALL_FORMS/' 

# ---------------------------------------------------------
# 1. DATA LOADING
# ---------------------------------------------------------
def load_data(filename):
    """Safely load a CSV file into a pandas DataFrame."""
    file_path = os.path.join(DATA_DIR, filename)
    try:
        print(f"Loading {filename}...")
        return pd.read_csv(file_path, low_memory=False)
    except FileNotFoundError:
        print(f"  -> Warning: {filename} not found. Skipping.")
        return None

print("=== PART 1: LOADING DATASETS ===")
datasets = {
    'demographics': load_data('F_PROACT_DEMOGRAPHICS.csv'),
    'alsfrs': load_data('F_PROACT_ALSFRS.csv'),
    'death': load_data('F_PROACT_DEATHDATA.csv'),
    'treatment': load_data('F_PROACT_TREATMENT.csv'),
    'riluzole': load_data('F_PROACT_RILUZOLE.csv'),
    'fvc': load_data('F_PROACT_FVC.csv'),
    'svc': load_data('F_PROACT_SVC.csv'),
    'vitals': load_data('F_PROACT_VITALSIGNS.csv'),
    'handgrip': load_data('F_PROACT_HANDGRIPSTRENGTH.csv'),
    'muscle': load_data('F_PROACT_MUSCLESTRENGTH.csv'),
    'neurofilament': load_data('F_PROACT_Neurofilament.csv'),
    'adverse_events': load_data('F_PROACT_ADVERSEEVENTS.csv'),
    'history': load_data('F_PROACT_ALSHISTORY.csv'),
    'conmeds': load_data('F_PROACT_CONMEDS.csv'),
    'elescorial': load_data('F_PROACT_ELESCORIAL.csv'),
    'family': load_data('F_PROACT_FAMILYHISTORY.csv'),
    'labs': load_data('F_PROACT_LABS.csv')
}


=== PART 1: LOADING DATASETS ===
Loading F_PROACT_DEMOGRAPHICS.csv...
Loading F_PROACT_ALSFRS.csv...
Loading F_PROACT_DEATHDATA.csv...
Loading F_PROACT_TREATMENT.csv...
Loading F_PROACT_RILUZOLE.csv...
Loading F_PROACT_FVC.csv...
Loading F_PROACT_SVC.csv...
Loading F_PROACT_VITALSIGNS.csv...
Loading F_PROACT_HANDGRIPSTRENGTH.csv...
Loading F_PROACT_MUSCLESTRENGTH.csv...
Loading F_PROACT_Neurofilament.csv...
Loading F_PROACT_ADVERSEEVENTS.csv...
Loading F_PROACT_ALSHISTORY.csv...
Loading F_PROACT_CONMEDS.csv...
Loading F_PROACT_ELESCORIAL.csv...
Loading F_PROACT_FAMILYHISTORY.csv...
Loading F_PROACT_LABS.csv...


In [26]:
datasets["alsfrs"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81229 entries, 0 to 81228
Data columns (total 20 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   subject_id                       81229 non-null  int64  
 1   Q1_Speech                        79926 non-null  float64
 2   Q2_Salivation                    79924 non-null  float64
 3   Q3_Swallowing                    79923 non-null  float64
 4   Q4_Handwriting                   79920 non-null  float64
 5   Q5a_Cutting_without_Gastrostomy  74348 non-null  float64
 6   Q5b_Cutting_with_Gastrostomy     6367 non-null   float64
 7   Q6_Dressing_and_Hygiene          79917 non-null  float64
 8   Q7_Turning_in_Bed                79916 non-null  float64
 9   Q8_Walking                       79920 non-null  float64
 10  Q9_Climbing_Stairs               79922 non-null  float64
 11  Q10_Respiratory                  36384 non-null  float64
 12  ALSFRS_Delta      